In [7]:
#pip install Pillow exifread simplekml folium

In [ ]:
"""
Drone Image Geotag Extractor - Trajectory-Based Coloring
========================================================
Reads GPS EXIF metadata from drone images in a folder and creates:
1. Interactive HTML map with satellite imagery basemap (Leaflet)
2. KMZ file for Google Earth with embedded graphics

Features:
- Segments trajectories by 5-minute time gaps
- Each trajectory has a unique color for both markers and lines
- Elevation used ONLY for 3D Z-values, NOT for symbology

Author: Farid Javadnejad
Date: 2026-04-24
"""

import os
import io
import zipfile
from datetime import datetime
from PIL import Image, ImageDraw
from PIL.ExifTags import TAGS, GPSTAGS
import folium
from folium import plugins
import simplekml
import numpy as np

# Import elevation library for DEM fallback
try:
    import srtm
    SRTM_AVAILABLE = True
except ImportError:
    SRTM_AVAILABLE = False
    print("INFO: 'srtm.py' library not found. DEM fallback will not be available.")
    print("      Install with: pip install srtm.py")

# ============================================================================
# CONFIGURATION
# ============================================================================

# Input folder containing drone images
INPUT_FOLDER = r"C:\Users\USFJ139860\OneDrive - WSP O365\Projects\260422 - US82 UAS Photoshoot\Images"

# Output files (saved in same folder as images)
OUTPUT_HTML = os.path.join(INPUT_FOLDER, "drone_image_map.html")
OUTPUT_KMZ = os.path.join(INPUT_FOLDER, "drone_image_locations.kmz")

# Trajectory segmentation threshold (seconds)
TIME_GAP_THRESHOLD = 300  # 5 minutes

# Trajectory colors (applied to both markers and lines)
# Colors are reused cyclically if there are more trajectories than colors
TRAJECTORY_COLORS = [
    '#FF0000',  # Red
    '#00FF00',  # Green
    '#0000FF',  # Blue
    '#FFFF00',  # Yellow
    '#FF00FF',  # Magenta
    '#00FFFF',  # Cyan
    '#FF8800',  # Orange
    '#8800FF',  # Purple
    '#00FF88',  # Spring Green
    '#FF0088'   # Deep Pink
]

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def convert_to_degrees(value):
    """
    Convert GPS coordinates from EXIF format to decimal degrees.
    
    Args:
        value: GPS coordinate in EXIF format (degrees, minutes, seconds)
    
    Returns:
        float: Decimal degrees
    """
    try:
        d = float(value[0])
        m = float(value[1])
        s = float(value[2])
        return d + (m / 60.0) + (s / 3600.0)
    except:
        return None


def parse_exif_datetime(datetime_str):
    """
    Parse EXIF datetime string to Python datetime object.
    
    Args:
        datetime_str (str): EXIF datetime string (format: "YYYY:MM:DD HH:MM:SS")
    
    Returns:
        datetime: Python datetime object or None if parsing fails
    """
    try:
        # EXIF format: "2024:04:15 14:23:45"
        return datetime.strptime(datetime_str, "%Y:%m:%d %H:%M:%S")
    except:
        return None


def get_ground_elevation_fallback(latitude, longitude):
    """
    Retrieve ground elevation (terrain height above MSL) from DEM as fallback.
    Uses SRTM (Shuttle Radar Topography Mission) 90m DEM data.
    
    Args:
        latitude (float): Latitude in decimal degrees
        longitude (float): Longitude in decimal degrees
    
    Returns:
        float: Ground elevation in meters above MSL, or None if unavailable
    """
    try:
        if SRTM_AVAILABLE:
            elevation_data = srtm.get_data()
            elev = elevation_data.get_elevation(latitude, longitude)
            return elev if elev is not None else None
        return None
    except Exception as e:
        print(f"  ⚠ Error retrieving DEM elevation for ({latitude:.6f}, {longitude:.6f}): {str(e)}")
        return None


def get_gps_data(image_path):
    """
    Extract GPS location data from an image file.
    
    ELEVATION PRECEDENCE (used ONLY for Z-values, NOT symbology):
    1. PRIMARY: EXIF GPSAltitude (absolute elevation recorded by drone, AMSL)
    2. FALLBACK: Ground elevation from SRTM DEM if GPSAltitude is missing
    
    Args:
        image_path (str): Path to the image file
    
    Returns:
        dict: Dictionary containing GPS metadata
    """
    result = {
        'latitude': None,
        'longitude': None,
        'absolute_elevation': None,  # PRIMARY: EXIF GPSAltitude or FALLBACK: DEM ground elevation
        'elevation_source': None,    # 'EXIF' or 'DEM' to track data source
        'filename': os.path.basename(image_path),
        'timestamp': None,
        'datetime_obj': None,  # For sorting and trajectory segmentation
        'trajectory_id': None  # Assigned during trajectory segmentation
    }
    
    try:
        with Image.open(image_path) as img:
            exif_data = img._getexif()
            
            if exif_data:
                # Extract GPS info
                gps_info = {}
                for tag, value in exif_data.items():
                    tag_name = TAGS.get(tag, tag)
                    
                    if tag_name == 'GPSInfo':
                        for gps_tag in value:
                            gps_tag_name = GPSTAGS.get(gps_tag, gps_tag)
                            gps_info[gps_tag_name] = value[gps_tag]
                    
                    # Extract timestamp
                    if tag_name == 'DateTimeOriginal':
                        result['timestamp'] = str(value)
                        result['datetime_obj'] = parse_exif_datetime(str(value))
                
                # Process GPS coordinates
                if 'GPSLatitude' in gps_info and 'GPSLongitude' in gps_info:
                    lat = convert_to_degrees(gps_info['GPSLatitude'])
                    lon = convert_to_degrees(gps_info['GPSLongitude'])
                    
                    # Handle N/S and E/W
                    if gps_info.get('GPSLatitudeRef') == 'S':
                        lat = -lat
                    if gps_info.get('GPSLongitudeRef') == 'W':
                        lon = -lon
                    
                    result['latitude'] = lat
                    result['longitude'] = lon
                    
                    # PRIMARY: Extract EXIF GPSAltitude (drone-recorded absolute elevation)
                    if 'GPSAltitude' in gps_info:
                        altitude = gps_info['GPSAltitude']
                        if isinstance(altitude, tuple):
                            result['absolute_elevation'] = float(altitude[0]) / float(altitude[1])
                        else:
                            result['absolute_elevation'] = float(altitude)
                        result['elevation_source'] = 'EXIF'
                    
                    # FALLBACK: If no EXIF altitude, retrieve ground elevation from DEM
                    if result['absolute_elevation'] is None:
                        dem_elevation = get_ground_elevation_fallback(lat, lon)
                        if dem_elevation is not None:
                            result['absolute_elevation'] = dem_elevation
                            result['elevation_source'] = 'DEM'
    
    except Exception as e:
        print(f"Error processing {image_path}: {str(e)}")
    
    return result


def process_images(folder_path):
    """
    Process all images in a folder and extract GPS metadata with absolute elevation.
    
    Args:
        folder_path (str): Path to folder containing images
    
    Returns:
        list: List of dictionaries containing image GPS metadata, sorted by timestamp
    """
    image_data = []
    
    # Supported image formats
    supported_formats = ('.jpg', '.jpeg', '.JPG', '.JPEG')
    
    print(f"Scanning folder: {folder_path}")
    
    if not os.path.exists(folder_path):
        print(f"ERROR: Folder not found: {folder_path}")
        return image_data
    
    # Iterate through all files in folder
    for filename in os.listdir(folder_path):
        if filename.endswith(supported_formats):
            file_path = os.path.join(folder_path, filename)
            print(f"Processing: {filename}")
            
            metadata = get_gps_data(file_path)
            
            # Only include images with valid GPS coordinates
            if metadata['latitude'] and metadata['longitude']:
                metadata['file_path'] = file_path
                image_data.append(metadata)
                
                if metadata['absolute_elevation'] is not None:
                    elev_str = f"{metadata['absolute_elevation']:.1f} m ({metadata['elevation_source']})"
                else:
                    elev_str = "N/A"
                
                time_str = metadata['timestamp'] or "N/A"
                print(f"  ✓ Lat: {metadata['latitude']:.6f}, Lon: {metadata['longitude']:.6f}, Elev: {elev_str}, Time: {time_str}")
            else:
                print(f"  ✗ No GPS data found")
    
    # Sort images by timestamp (images without timestamp go to end)
    image_data.sort(key=lambda x: (x['datetime_obj'] is None, x['datetime_obj'], x['filename']))
    
    print(f"\nTotal images with GPS data: {len(image_data)}")
    
    # Statistics on elevation sources
    exif_count = sum(1 for img in image_data if img['elevation_source'] == 'EXIF')
    dem_count = sum(1 for img in image_data if img['elevation_source'] == 'DEM')
    none_count = sum(1 for img in image_data if img['elevation_source'] is None)
    
    print(f"Elevation sources: {exif_count} from EXIF, {dem_count} from DEM, {none_count} missing")
    
    return image_data


def segment_trajectories(image_data, time_gap_seconds=300):
    """
    Segment images into separate trajectories based on time gaps.
    Each trajectory is assigned a unique ID and color.
    
    TIME GAP THRESHOLD: If the time difference between consecutive images 
    exceeds time_gap_seconds (default: 5 minutes), a new trajectory begins.
    
    Args:
        image_data (list): List of image metadata dictionaries (sorted by time)
        time_gap_seconds (int): Maximum time gap in seconds before starting new trajectory
    
    Returns:
        list: List of trajectories, where each trajectory is a list of image metadata
    """
    if not image_data:
        return []
    
    trajectories = []
    current_trajectory = [image_data[0]]
    trajectory_id = 0
    
    # Assign first image to trajectory 0
    image_data[0]['trajectory_id'] = trajectory_id
    
    for i in range(1, len(image_data)):
        prev_img = image_data[i - 1]
        curr_img = image_data[i]
        
        # Check if both images have valid timestamps
        if prev_img['datetime_obj'] and curr_img['datetime_obj']:
            time_diff = (curr_img['datetime_obj'] - prev_img['datetime_obj']).total_seconds()
            
            # If time gap exceeds threshold (5 minutes), start new trajectory
            if time_diff > time_gap_seconds:
                trajectories.append(current_trajectory)
                trajectory_id += 1
                current_trajectory = [curr_img]
                curr_img['trajectory_id'] = trajectory_id
                print(f"  → New trajectory {trajectory_id} started: {time_diff:.0f}s gap between {prev_img['filename']} and {curr_img['filename']}")
            else:
                curr_img['trajectory_id'] = trajectory_id
                current_trajectory.append(curr_img)
        else:
            # If either image lacks timestamp, keep in same trajectory but warn
            curr_img['trajectory_id'] = trajectory_id
            current_trajectory.append(curr_img)
            if not curr_img['datetime_obj']:
                print(f"  ⚠ No timestamp for {curr_img['filename']}, grouped by file order")
    
    # Add the last trajectory
    trajectories.append(current_trajectory)
    
    print(f"\nSegmented into {len(trajectories)} trajectory/trajectories:")
    for idx, traj in enumerate(trajectories):
        # Assign color to trajectory (cyclical reuse if needed)
        color = TRAJECTORY_COLORS[idx % len(TRAJECTORY_COLORS)]
        print(f"  Trajectory {idx + 1}: {len(traj)} images, Color: {color}")
    
    return trajectories


def create_circle_icon(color, size=40):
    """
    Create a circular marker icon as PNG image data.
    
    Args:
        color (str): Hex color code (e.g., '#FF0000')
        size (int): Icon size in pixels
    
    Returns:
        bytes: PNG image data
    """
    # Create a new image with transparency
    img = Image.new('RGBA', (size, size), (0, 0, 0, 0))
    draw = ImageDraw.Draw(img)
    
    # Draw white border circle
    draw.ellipse([2, 2, size-2, size-2], fill=color, outline='white', width=3)
    
    # Draw small center dot
    center = size // 2
    draw.ellipse([center-3, center-3, center+3, center+3], fill='white', outline=color, width=2)
    
    # Convert to PNG bytes
    img_bytes = io.BytesIO()
    img.save(img_bytes, format='PNG')
    return img_bytes.getvalue()


def create_leaflet_map(image_data, trajectories, output_file):
    """
    Create an interactive Leaflet map with satellite imagery basemap.
    Markers and lines are colored by trajectory (NOT by elevation).
    
    Args:
        image_data (list): List of all image metadata dictionaries
        trajectories (list): List of trajectories (each is a list of image metadata)
        output_file (str): Output HTML file path
    """
    if not image_data:
        print("No data to map!")
        return
    
    # Calculate map center (average of all coordinates)
    avg_lat = sum(img['latitude'] for img in image_data) / len(image_data)
    avg_lon = sum(img['longitude'] for img in image_data) / len(image_data)
    
    # Create base map (no default tiles, we'll add custom ones)
    m = folium.Map(
        location=[avg_lat, avg_lon],
        zoom_start=16,
        tiles=None
    )
    
    # Add Esri World Imagery (satellite ONLY - no labels) - DEFAULT
    folium.TileLayer(
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
        attr='Esri',
        name='Satellite',
        overlay=False,
        control=True,
        show=True  # Default visible layer
    ).add_to(m)
    
    # Add street map basemap - OPTIONAL
    folium.TileLayer(
        tiles='OpenStreetMap',
        name='Street Map',
        overlay=False,
        control=True,
        show=False
    ).add_to(m)
    
    # Add circle markers for each image
    # Color is determined by trajectory ID, NOT by elevation
    for idx, img in enumerate(image_data):
        # Get trajectory color (cyclical reuse)
        trajectory_id = img['trajectory_id'] if img['trajectory_id'] is not None else 0
        color = TRAJECTORY_COLORS[trajectory_id % len(TRAJECTORY_COLORS)]
        
        # Create popup content with metadata
        elev_display = f"{img['absolute_elevation']:.1f}" if img['absolute_elevation'] is not None else "N/A"
        elev_source = img['elevation_source'] or "N/A"
        
        popup_html = f"""
        <div style="width: 300px;">
            <h4 style="margin-bottom: 10px;">{img['filename']}</h4>
            <table style="width:100%; font-size: 12px;">
                <tr><td><b>Latitude:</b></td><td>{img['latitude']:.6f}°</td></tr>
                <tr><td><b>Longitude:</b></td><td>{img['longitude']:.6f}°</td></tr>
                <tr><td><b>Absolute Elevation:</b></td><td>{elev_display} m</td></tr>
                <tr><td><b>Elevation Source:</b></td><td>{elev_source}</td></tr>
                <tr><td><b>Trajectory:</b></td><td>{trajectory_id + 1}</td></tr>
                <tr><td><b>Trajectory Color:</b></td><td><span style="background-color:{color}; padding:2px 10px; color:white;">&nbsp;&nbsp;&nbsp;</span></td></tr>
                <tr><td><b>Timestamp:</b></td><td>{img['timestamp'] or 'N/A'}</td></tr>
            </table>
        </div>
        """
        
        # Create circle marker with trajectory color
        folium.CircleMarker(
            location=[img['latitude'], img['longitude']],
            radius=8,
            popup=folium.Popup(popup_html, max_width=320),
            tooltip=f"{img['filename']} (Trajectory {trajectory_id + 1})",
            color='white',
            fillColor=color,
            fillOpacity=0.8,
            weight=2
        ).add_to(m)
    
    # Add flight path polylines - one per trajectory with matching color
    for traj_idx, trajectory in enumerate(trajectories):
        if len(trajectory) > 1:
            flight_path = [[img['latitude'], img['longitude']] for img in trajectory]
            
            # Use trajectory color (cyclical reuse)
            trajectory_color = TRAJECTORY_COLORS[traj_idx % len(TRAJECTORY_COLORS)]
            
            folium.PolyLine(
                flight_path,
                color=trajectory_color,
                weight=3,
                opacity=0.8,
                name=f'Trajectory {traj_idx + 1} ({len(trajectory)} images)'
            ).add_to(m)
    
    # Add layer control
    folium.LayerControl(position='topright').add_to(m)
    
    # Add fullscreen button
    plugins.Fullscreen().add_to(m)
    
    # Add measurement tool
    plugins.MeasureControl(position='topleft').add_to(m)
    
    # Save map
    m.save(output_file)
    print(f"\n✓ Interactive map saved: {output_file}")


def create_kmz_file(image_data, trajectories, output_file):
    """
    Create a KMZ file with image locations and flight paths.
    Placemarks and lines are colored by trajectory (NOT by elevation).
    Elevation is used ONLY for 3D Z-values.
    
    Args:
        image_data (list): List of all image metadata dictionaries
        trajectories (list): List of trajectories (each is a list of image metadata)
        output_file (str): Output KMZ file path
    """
    if not image_data:
        print("No data to export to KMZ!")
        return
    
    # Create KML document
    kml = simplekml.Kml()
    
    # Create icon files for each trajectory color
    # Generate icons for all possible trajectories (limited to unique colors)
    num_unique_colors = min(len(trajectories), len(TRAJECTORY_COLORS))
    icon_files = {}
    
    for i in range(num_unique_colors):
        color = TRAJECTORY_COLORS[i % len(TRAJECTORY_COLORS)]
        icon_data = create_circle_icon(color, size=40)
        icon_filename = f'icon_trajectory_{i+1}.png'
        icon_files[icon_filename] = icon_data
        
        # Create KML style
        style = simplekml.Style()
        style.iconstyle.icon.href = icon_filename
        style.iconstyle.scale = 1.0
        kml.document.styles.append(style)
    
    # Create a folder for all drone images
    points_folder = kml.newfolder(name='Drone Image Locations')
    
    for img in image_data:
        # Create a point for each image
        pnt = points_folder.newpoint(name=img['filename'])
        
        # Use absolute elevation for Z-value (3D positioning in Google Earth)
        z_value = img['absolute_elevation'] if img['absolute_elevation'] is not None else 0
        pnt.coords = [(img['longitude'], img['latitude'], z_value)]
        
        # Apply trajectory-based icon (NOT elevation-based)
        trajectory_id = img['trajectory_id'] if img['trajectory_id'] is not None else 0
        icon_idx = trajectory_id % num_unique_colors
        icon_filename = f'icon_trajectory_{icon_idx+1}.png'
        pnt.style.iconstyle.icon.href = icon_filename
        pnt.style.iconstyle.scale = 1.0
        
        # Add description with metadata
        elev_display = f"{img['absolute_elevation']:.1f}" if img['absolute_elevation'] is not None else "N/A"
        elev_source = img['elevation_source'] or "N/A"
        trajectory_color = TRAJECTORY_COLORS[trajectory_id % len(TRAJECTORY_COLORS)]
        
        description = f"""
        <![CDATA[
        <table border="1" cellpadding="5">
            <tr><th>Property</th><th>Value</th></tr>
            <tr><td>Filename</td><td>{img['filename']}</td></tr>
            <tr><td>Latitude</td><td>{img['latitude']:.6f}°</td></tr>
            <tr><td>Longitude</td><td>{img['longitude']:.6f}°</td></tr>
            <tr><td>Absolute Elevation</td><td>{elev_display} m</td></tr>
            <tr><td>Elevation Source</td><td>{elev_source}</td></tr>
            <tr><td>Trajectory</td><td>{trajectory_id + 1}</td></tr>
            <tr><td>Trajectory Color</td><td><span style="background-color:{trajectory_color}; padding:2px 10px; color:white;">&nbsp;&nbsp;&nbsp;</span></td></tr>
            <tr><td>Timestamp</td><td>{img['timestamp'] or 'N/A'}</td></tr>
        </table>
        ]]>
        """
        pnt.description = description
        
        # Set altitude mode to absolute (relative to sea level)
        pnt.altitudemode = simplekml.AltitudeMode.absolute
    
    # Create a folder for flight paths
    paths_folder = kml.newfolder(name='Flight Paths')
    
    # Create one linestring per trajectory with matching trajectory color
    for traj_idx, trajectory in enumerate(trajectories):
        if len(trajectory) > 1:
            # Use absolute elevation for Z-values in flight path
            coords = [
                (img['longitude'], img['latitude'], img['absolute_elevation'] if img['absolute_elevation'] is not None else 0) 
                for img in trajectory
            ]
            
            # Use trajectory color (cyclical reuse)
            trajectory_color = TRAJECTORY_COLORS[traj_idx % len(TRAJECTORY_COLORS)]
            
            # Convert hex color to KML color (AABBGGRR format)
            hex_color = trajectory_color.lstrip('#')
            # Reverse RGB to BGR for KML
            kml_color = f'ff{hex_color[4:6]}{hex_color[2:4]}{hex_color[0:2]}'
            
            linestring = paths_folder.newlinestring(name=f'Trajectory {traj_idx + 1}')
            linestring.coords = coords
            linestring.style.linestyle.color = kml_color
            linestring.style.linestyle.width = 4
            linestring.altitudemode = simplekml.AltitudeMode.absolute
            
            # Add description with trajectory info
            start_time = trajectory[0]['timestamp'] or 'N/A'
            end_time = trajectory[-1]['timestamp'] or 'N/A'
            
            # Calculate elevation statistics for reference
            elevations_in_traj = [img['absolute_elevation'] for img in trajectory if img['absolute_elevation'] is not None]
            if elevations_in_traj:
                avg_elev = np.mean(elevations_in_traj)
                min_elev = min(elevations_in_traj)
                max_elev = max(elevations_in_traj)
            else:
                avg_elev = min_elev = max_elev = 0
            
            linestring.description = f"""
            <![CDATA[
            <b>Trajectory {traj_idx + 1}</b><br/>
            Color: <span style="background-color:{trajectory_color}; padding:2px 10px; color:white;">&nbsp;&nbsp;&nbsp;</span><br/>
            Images: {len(trajectory)}<br/>
            Average Elevation: {avg_elev:.1f} m<br/>
            Elevation Range: {min_elev:.1f} - {max_elev:.1f} m<br/>
            Start: {start_time}<br/>
            End: {end_time}
            ]]>
            """
    
    # Save KML to temporary location
    temp_kml_path = output_file.replace('.kmz', '_temp.kml')
    kml.save(temp_kml_path)
    
    # Create KMZ (ZIP archive containing KML + icon files)
    with zipfile.ZipFile(output_file, 'w', zipfile.ZIP_DEFLATED) as kmz:
        # Add KML file as doc.kml (required name for KMZ)
        kmz.write(temp_kml_path, 'doc.kml')
        
        # Add icon files
        for icon_filename, icon_data in icon_files.items():
            kmz.writestr(icon_filename, icon_data)
    
    # Clean up temporary KML file
    os.remove(temp_kml_path)
    
    print(f"✓ KMZ file saved: {output_file}")
    print(f"  - Embedded {len(icon_files)} custom icon files")


# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    """
    Main execution function.
    """
    print("=" * 70)
    print("DRONE IMAGE GEOTAG EXTRACTOR - TRAJECTORY-BASED COLORING")
    print("=" * 70)
    print()
    
    # Process images and retrieve absolute elevation (EXIF primary, DEM fallback)
    image_data = process_images(INPUT_FOLDER)
    
    if not image_data:
        print("\n⚠ No images with GPS data found. Please check:")
        print("  1. Folder path is correct")
        print("  2. Images contain EXIF GPS metadata")
        print("  3. Images are in JPEG format")
        return
    
    # Segment trajectories by time gaps (5 minutes)
    print("\n" + "=" * 70)
    print("SEGMENTING TRAJECTORIES (5-minute threshold)")
    print("=" * 70)
    trajectories = segment_trajectories(image_data, TIME_GAP_THRESHOLD)
    
    # Generate outputs
    print("\n" + "=" * 70)
    print("GENERATING OUTPUTS")
    print("=" * 70)
    
    create_leaflet_map(image_data, trajectories, OUTPUT_HTML)
    create_kmz_file(image_data, trajectories, OUTPUT_KMZ)
    
    print("\n" + "=" * 70)
    print("PROCESS COMPLETE!")
    print("=" * 70)
    print(f"\nOutputs created in: {INPUT_FOLDER}")
    print(f"  • HTML Map: {OUTPUT_HTML}")
    print(f"  • KMZ File: {OUTPUT_KMZ}")
    print(f"\nTotal images processed: {len(image_data)}")
    print(f"Total trajectories: {len(trajectories)}")
    print(f"Time gap threshold: {TIME_GAP_THRESHOLD} seconds (5 minutes)")
    print(f"\nNote: Coloring is based on TRAJECTORY, not elevation.")
    print(f"      Elevation is used ONLY for 3D Z-values in KMZ.")


if __name__ == "__main__":
    main()

INFO: 'srtm.py' library not found. DEM fallback will not be available.
      Install with: pip install srtm.py
DRONE IMAGE GEOTAG EXTRACTOR - TRAJECTORY-BASED COLORING

Scanning folder: C:\Users\USFJ139860\OneDrive - WSP O365\Projects\260422 - US82 UAS Photoshoot\Photo_08
Processing: DJI_20260422113921_0053_D.JPG
  ✓ Lat: 32.959114, Lon: -103.348390, Elev: 1338.7 m (EXIF), Time: 2026:04:22 11:39:21
Processing: DJI_20260422113932_0054_D.JPG
  ✓ Lat: 32.959107, Lon: -103.348385, Elev: 1395.9 m (EXIF), Time: 2026:04:22 11:39:32
Processing: DJI_20260422113943_0055_D.JPG
  ✓ Lat: 32.959163, Lon: -103.348828, Elev: 1411.7 m (EXIF), Time: 2026:04:22 11:39:43
Processing: DJI_20260422113948_0056_D.JPG
  ✓ Lat: 32.959486, Lon: -103.348754, Elev: 1411.6 m (EXIF), Time: 2026:04:22 11:39:48
Processing: DJI_20260422113957_0057_D.JPG
  ✓ Lat: 32.960225, Lon: -103.349007, Elev: 1411.7 m (EXIF), Time: 2026:04:22 11:39:57
Processing: DJI_20260422114001_0058_D.JPG
  ✓ Lat: 32.960436, Lon: -103.349296, El